In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv("/kaggle/input/heart-disease-indicators/heart_disease_health_indicators.csv")

**EDA**

In [ ]:
df.columns


In [ ]:
print(df.describe())


In [ ]:
print(df.isnull().sum())


We can see that our dataframe is clean already. Thank's nice, thanks!

In [ ]:
print(df.dtypes)



In [ ]:
print(df['HeartDiseaseorAttack'].value_counts())

In [ ]:
print(df['Sex'].value_counts())


In [ ]:
print(df['Age'].value_counts())

In [ ]:
print(df['Education'].value_counts())

<h1> **Some nice visualisations** </h1>

In [ ]:
plt.figure(figsize=(16,12))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', linewidths=.5)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Counts of Sex')
plt.show()

In [ ]:
sns.countplot(data=df, x='HeartDiseaseorAttack')
plt.title('Heart disease')
plt.show()

In [ ]:
subset_cols = ['Age', 'BMI', 'HighChol','HeartDiseaseorAttack']
sns.pairplot(df[subset_cols])
plt.show()

In [ ]:
sns.histplot(data=df, x='Age', hue='HeartDiseaseorAttack', bins=30, kde=True, element='step')
plt.title('Age Distribution by Heart Disease or Attack')
plt.show()


In [ ]:
# Define custom labels
labels = ['Female', 'Male']

# Create the plot
sns.countplot(data=df, x='Sex', hue='HeartDiseaseorAttack')

# Update the x-axis labels
plt.xticks(ticks=[0,1], labels=labels)
plt.title('Heart Disease or Attack Counts by Sex')
plt.xlabel('Sex')  # This makes sure the x-axis label is clear
plt.legend(title='Heart Disease or Attack', labels=['No', 'Yes'])  # This makes the legend more intuitive

# Display the plot
plt.show()

In [ ]:
sns.histplot(data=df, x='BMI', hue='HeartDiseaseorAttack', bins=30, kde=True, element='step')
plt.title('BMI Distribution by Heart Disease or Attack')
plt.show()

In [ ]:
sns.countplot(data=df, x='Smoker', hue='HeartDiseaseorAttack')
plt.title('Heart Disease or Attack Counts by Smoking Status')
plt.show()

In [ ]:
sns.countplot(data=df, x='PhysActivity', hue='HeartDiseaseorAttack')
plt.title('Heart Disease or Attack Counts by Physical Activity Level')
plt.show()

<H1> PREDICTIONS </H1>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Split the data
X = df.drop('HeartDiseaseorAttack', axis=1)
y = df['HeartDiseaseorAttack']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

# Create transformers for both numerical and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers into a preprocessor step
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Combine preprocessor and model into one pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', LogisticRegression())])

# Train the model
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluate the model
print("Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Heart Disease", "Heart Disease"])
disp.plot()
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imPipeline

# Create the SMOTE object
smote = SMOTE(random_state=42)

# Modify your pipeline to include SMOTE for the training data:
pipeline = imPipeline(steps=[('preprocessor', preprocessor),
                             ('smote', smote),
                             ('classifier', LogisticRegression())])


# Train the model
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluate the model
print("Accuracy: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Heart Disease", "Heart Disease"])
disp.plot()
plt.show()

Performance for Class 0 (No Heart Disease):

We see that the precision is now 0.97(quite high) but the recall has dropped to 0.75
This implies that while the model is very accurate when predicting no heart disease, it is missing a quarter of the true negatives.

Recall for Class 1 (Heart Disease): 

This indicates that the model identifies 78% of the true heart disease cases, which is a significant improvement from before SMOTE was applied. However, this comes at the expense of precision, as seen above.

Precision for Class 1 (Heart Disease): 0.25

This implies that when the model predicts that someone has heart disease, it is correct 25% of the time. This is low precision, meaning there's a high false positive rate. In practical terms, many people are being flagged as having heart disease when they actually don't. However, it is a very good warning system for someone who is at risk. Let's try and do more!

In [ ]:
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/kaggle/input/heart-disease-indicators/heart_disease_health_indicators.csv")

X = df.drop('HeartDiseaseorAttack', axis=1)
y = df['HeartDiseaseorAttack']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Initialize and train the Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_smote, y_train_smote)

# Predictions
y_pred = rf_classifier.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Heart Disease", "Heart Disease"])
disp.plot()
plt.show()


This means that when the Random Forest model predicts that someone has heart disease, it is correct  23% of the time. This precision is somewhat comparable to what we observed with the logistic regression after applying SMOTE.

In [ ]:
import xgboost as xgb
from imblearn.pipeline import make_pipeline

# Create a pipeline with SMOTE and XGBClassifier
pipeline = make_pipeline(
    SMOTE(random_state=42),
    xgb.XGBClassifier(objective='binary:logistic', random_state=42)
)

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Heart Disease", "Heart Disease"])
disp.plot()
plt.show()

The XGBClassifier provides a decent recall of 67%, meaning it can identify a significant portion of the true heart disease cases. If the primary aim is to capture most cases, even at the expense of some false positives, this might be acceptable.
However, with a precision of 21%, there's a risk of misclassifying a significant number of people who don't have heart disease as having it. Depending on the application, this might result in unnecessary tests or interventions.


*LET'S GO WITH A NEURAL NETWORK!*

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Assuming df is your dataframe
X = df.drop('HeartDiseaseorAttack', axis=1)
y = df['HeartDiseaseorAttack']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [ ]:
model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train_smote.shape[1],)),
    layers.Dropout(0.5),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# Training the model
history = model.fit(
    X_train_smote, y_train_smote,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)
# Evaluate on the test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)
print("Test Accuracy: ", accuracy)

# Predictions
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int).flatten()


In [ ]:
# Generate the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix using seaborn
plt.figure(figsize=(8,6))
sns.heatmap(conf_matrix, annot=True, fmt='g', cmap='Blues', cbar=False)
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()

# Generate and print the classification report
print(classification_report(y_test, y_pred))


The neural network does a decent job identifying those without a heart attack (Class 0) but struggles to precisely identify those with a heart attack (Class 1) without making a significant number of false alarms. Given the critical nature of the prediction, while recall for Class 1 is more important (and it's decent at 67%), the precision for Class 1 (at 29%) suggests the model might be overly cautious, flagging many as potential heart attack risks who aren't.

<H1>**RESULTS**</H1>

Logistic Regression has the highest recall, making it the top performer in identifying the maximum number of potential heart attack cases. This model will ensure that 78% of the people who actually are at risk of a heart attack will be identified.
Logistic Regression with SMOTE appears to be the most effective tool in this analysis. It will correctly identify the highest percentage of people who are at risk of a heart attack, although there will be a significant number of false alarms. But since false positives are not a concern in this scenario, Logistic Regression stands out as the top choice.

**Some extra Neural Networks for fun UwU**

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Assuming you have already loaded your data into 'X' and 'y'
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Use SMOTE for over-sampling
smote = SMOTE()
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Model Architecture
model = keras.Sequential([
    keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal', input_shape=(X_train_smote.shape[1],)),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu', kernel_initializer='he_normal'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

# Learning rate scheduler
lr_schedule = keras.callbacks.LearningRateScheduler(
    lambda epoch: 1e-3 * 10**(epoch / 20))

# Early stopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train_smote, y_train_smote, 
                    validation_data=(X_test, y_test),
                    epochs=100, batch_size=32, 
                    callbacks=[lr_schedule, early_stopping])

# Predictions
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Classification report and confusion matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


The logistic regression (with SMOTE) has a higher recall (0.78) for predicting heart attacks compared to the neural network's recall of 0.70. This means the logistic regression model identifies a larger percentage of actual positive heart attack cases than the neural network.
Thee logistic regression (with SMOTE) appears to perform slightly better than the improved neural network in this dataset. However, our neural network has improved by a lot. Nice!

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Define the model
model = Sequential()

# Input layer
model.add(Dense(64, input_dim=X_train_smote.shape[1], activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Hidden layers
model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model
optimizer = Adam(learning_rate=0.001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Callbacks
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(X_train_smote, y_train_smote, validation_split=0.2, epochs=100, batch_size=32, callbacks=[reduce_lr, early_stopping])

y_pred_nn = (model.predict(X_test) > 0.5).astype("int32")

# Confusion matrix and classification report
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred_nn)
print(cm)
ConfusionMatrixDisplay(cm, display_labels=["No Heart Attack", "Heart Attack"]).plot()

print(classification_report(y_test, y_pred_nn))


Given these results, the enhanced neural network has improved from its previous iteration. However, logistic regression still slightly outperforms the neural network in terms of recall for identifying heart attacks.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Define the model
model_final = Sequential()

# Input layer
model_final.add(Dense(128, activation='relu', input_shape=(X_train_smote.shape[1],)))
model_final.add(BatchNormalization())
model_final.add(Dropout(0.5))

# Hidden layers
model_final.add(Dense(256, activation='relu'))
model_final.add(BatchNormalization())
model_final.add(Dropout(0.5))

model_final.add(Dense(256, activation='relu'))
model_final.add(BatchNormalization())
model_final.add(Dropout(0.5))

model_final.add(Dense(128, activation='relu'))
model_final.add(BatchNormalization())
model_final.add(Dropout(0.5))

# Output layer
model_final.add(Dense(1, activation='sigmoid'))

# Compile the model
model_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)

# Train the model
history_final = model_final.fit(X_train_smote, y_train_smote, 
                                validation_data=(X_test, y_test),
                                epochs=100, batch_size=64, 
                                callbacks=[early_stopping, reduce_lr])

# Evaluate the model
y_pred_final = (model_final.predict(X_test) > 0.5).astype("int32")
print(classification_report(y_test, y_pred_final))


The neural network and the logistic regression model using SMOTE have quite similar performances, with the logistic regression having a slight edge in recall. The recall for the logistic regression model was 0.78, and for the neural network, it's 0.71. Given the use case's objective, this metric is one of the most significant, and thus, the logistic regression model is performing slightly better.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# 1. Feature Engineering
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

scaler = StandardScaler()
X_train_poly = scaler.fit_transform(X_train_poly)
X_test_poly = scaler.transform(X_test_poly)

# 2. Model Architecture
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train_poly.shape[1],)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
optimizer = Adam(learning_rate=0.0001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# 3. Training Strategy
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train_poly, y_train, validation_data=(X_test_poly, y_test), epochs=100, batch_size=64, callbacks=[early_stopping])

# Prediction and evaluation
y_pred = (model.predict(X_test_poly) > 0.5).astype("int32")
print(classification_report(y_test, y_pred))


Considering the primary goal is to predict who has a heart attack, the model needs improvement, especially in terms of recall for class 1. High accuracy can be misleading in such cases where the dataset is imbalanced. The low recall means that the model is missing a large number of individuals who actually have a heart attack.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Define the model
model = Sequential()

# Input Layer
model.add(Dense(32, input_dim=X_train_scaled.shape[1], activation='relu'))
model.add(Dropout(0.5))

# Hidden Layers
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# Summary of the model architecture
model.summary()

# Train the model
history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test), epochs=100, batch_size=32, verbose=1)
from sklearn.metrics import classification_report, confusion_matrix

# Predict the results
y_pred = (model.predict(X_test_scaled) > 0.5).astype("int32")

# Classification report
print(classification_report(y_test, y_pred))

# Confusion matrix
confusion = confusion_matrix(y_test, y_pred)
print(confusion)


That's also bad! 
logistic with SMOTE won!